Tuesday: booked is not collected, and the join that lied

> "Booked revenue is not collected revenue. Some orders are paid in two instalments, some are
> refunded, some were never paid at all. Show me, order by order, what we actually collected
> against what we booked in Q2. If there is a gap, I want to know which orders and which
> channel."
>
> Anand Iyer, having read your Monday suite.

And one line from the platform lead, said in passing: the payments feed sometimes double-posts
when the gateway retries.

**MAP** two tables  ->  **DO** the naive join  ->  **SEE** the doubling  ->
**CHECK** the count habit  ->  **SUM** the report you would sign.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
sizes = kit.sql("SELECT 'orders' t, count(*) n FROM orders "
                "UNION ALL SELECT 'payments', count(*) FROM payments", conn=conn)
for r in sizes:
    print(f"{r['t']:>9}  {r['n']:,} rows")

   orders  1,000 rows
 payments  1,428 rows


## MAP. Two tables, and nothing yet connecting them

Booked lives in `orders`: an order was placed. Collected lives in `payments`: money arrived.
The join is the only thing that turns two facts into one number, and it is where the damage
happens.

In [2]:
kit.sequence(["orders", "payments"],
             [("orders", "payments", "match on order_id"),
              ("payments", "orders", "0, 1 or many rows back")],
             title="What a join actually asks for")

## DO. The total anybody would write first

One statement, both numbers, done. Read what comes back before reading the next cell.

In [3]:
naive = kit.sql("""
    SELECT sum(o.amount) AS booked, sum(p.amount) AS collected
    FROM orders o LEFT JOIN payments p ON p.order_id = o.order_id
""", conn=conn)[0]
book = kit.sql("SELECT sum(amount) s FROM orders", conn=conn)[0]["s"]
print("booked, over the join :", kit.rupees(naive["booked"]))
print("booked, from orders   :", kit.rupees(book))
print(f"\nratio: {float(naive['booked']) / float(book):.4f}")

booked, over the join : Rs 39,40,95,490
booked, from orders   : Rs 19,84,00,000

ratio: 1.9864


### Every row in that result is correct

Pick any row and check it. The order id is real, the amount is that order's real amount, the
payment is a genuine payment for that order. Nothing is corrupted and nothing is missing.

The total is twice the truth, and that combination is what makes this the most expensive
mistake in analytics.

In [4]:
rows = kit.sql("""SELECT count(*) n FROM orders o
                  LEFT JOIN payments p ON p.order_id = o.order_id""", conn=conn)[0]["n"]
kit.flow(["orders: 1,000", "LEFT JOIN", f"rows out: {rows:,}", f"+{rows - 1000}"],
         lit=[2], title="The count that was free to take and was not taken")
kit.check("the join returned more rows than it started with", rows > 1000, f"{rows:,} from 1,000")

## SEE. Where the extra rows came from

Nothing was duplicated in either table. The duplication happened in the join, because one order
can have several payment rows and every one of them makes a row.

In [5]:
kit.sql_table("""
    WITH doubled AS (
        SELECT order_id,
               CASE WHEN count(DISTINCT amount) = 1 THEN 'retry, the same amount twice'
                    ELSE 'instalment plan, different amounts' END AS kind
        FROM payments GROUP BY order_id HAVING count(*) > 1)
    SELECT kind, count(*) AS orders FROM doubled GROUP BY kind ORDER BY orders DESC
""", caption="450 orders carry more than one payment row", conn=conn)

kind,orders
"instalment plan, different amounts",400
"retry, the same amount twice",50


### Why 450 duplicates doubled a book of a thousand

The 450 are not a random slice. Large invoices get instalment terms, so the orders that were
duplicated are the orders that carry the revenue. Duplicate the biggest half of the book and
you do not inflate a total by five percent.

In [6]:
share = kit.sql("""
    WITH multi AS (SELECT order_id FROM payments GROUP BY order_id HAVING count(*) > 1)
    SELECT sum(CASE WHEN o.order_id IN (SELECT order_id FROM multi) THEN o.amount ELSE 0 END) AS in_multi,
           sum(o.amount) AS whole_book
    FROM orders o
""", conn=conn)[0]
pct = 100 * float(share["in_multi"]) / float(share["whole_book"])
print(f"the 450 duplicated orders carry {pct:.1f}% of the whole book")
kit.check("the duplicated orders carry most of the revenue", pct > 80, f"{pct:.1f}%")

the 450 duplicated orders carry 98.6% of the whole book


## SEE. What each join keeps

Each join type answers a different question about the rows that do not match. The difference
between INNER and LEFT here is exactly the thirty orders Anand asked about.

In [7]:
kit.sql_table("""
    SELECT 'inner' AS join_type, count(*) AS rows FROM orders o
      INNER JOIN payments p USING (order_id)
    UNION ALL
    SELECT 'left', count(*) FROM orders o LEFT JOIN payments p USING (order_id)
""", caption="Thirty rows apart, and those thirty are the question", conn=conn)

unpaid = kit.sql("""
    SELECT o.order_id, o.channel, o.amount FROM orders o
    LEFT JOIN payments p ON p.order_id = o.order_id
    WHERE p.payment_id IS NULL ORDER BY o.amount DESC
""", conn=conn)
kit.check("thirty orders were never paid", len(unpaid) == 30, f"{len(unpaid)} orders")
print("the two largest, both Q2:")
for r in unpaid[:2]:
    print(f"  {r['order_id']}  {r['channel']:>5}  {kit.rupees(r['amount'])}")

join_type,rows
inner,1420
left,1450


the two largest, both Q2:
  KR-00577  store  Rs 9,27,000
  KR-00582    web  Rs 7,70,000


### And the orphans on the other side

An anti-join run the other way finds money that arrived for something the book does not know
about. That is a question for the platform team rather than for you, and it still has to be
reported.

In [8]:
orphans = kit.sql("""
    SELECT p.payment_id, p.order_id, p.amount FROM payments p
    LEFT JOIN orders o ON o.order_id = p.order_id WHERE o.order_id IS NULL
""", conn=conn)
kit.check("payments exist whose order is not in the table", len(orphans) == 8, f"{len(orphans)}")
kit.table(["payment", "order it claims", "amount"],
          [[r["payment_id"], r["order_id"], kit.rupees(r["amount"])] for r in orphans],
          caption="Money in, order unknown")

payment,order it claims,amount
P-01421,KR-90000,"Rs 2,000"
P-01422,KR-90001,"Rs 2,310"
P-01423,KR-90002,"Rs 2,620"
P-01424,KR-90003,"Rs 2,930"
P-01425,KR-90004,"Rs 3,240"
P-01426,KR-90005,"Rs 3,550"
P-01427,KR-90006,"Rs 3,860"
P-01428,KR-90007,"Rs 4,170"


## CHECK. Collapse the many side, then join

A join is only done when its row count is explained: rows before, rows after, and a sentence
accounting for the difference. Aggregating to one row per key before joining makes the count
check pass by construction rather than by luck.

In [9]:
fixed = kit.sql("""
    WITH paid AS (SELECT order_id, sum(amount) AS collected FROM payments GROUP BY order_id)
    SELECT count(*) AS rows_out, sum(o.amount) AS booked,
           coalesce(sum(p.collected), 0) AS collected
    FROM orders o LEFT JOIN paid p ON p.order_id = o.order_id
""", conn=conn)[0]
print("rows out :", f"{fixed['rows_out']:,}")
print("booked   :", kit.rupees(fixed["booked"]))
print("collected:", kit.rupees(fixed["collected"]))
print("gap      :", kit.rupees(fixed["booked"] - fixed["collected"]))
kit.check("one row in, one row out", fixed["rows_out"] == 1000, f"{fixed['rows_out']:,}")
kit.check("booked now matches the orders table on its own", fixed["booked"] == book)

rows out : 1,000
booked   : Rs 19,84,00,000
collected: Rs 19,66,82,820
gap      : Rs 17,17,180


## SUM. The gap Anand asked about, split by channel

`coalesce` is doing real work below. An order with no payment joins to NULL, and NULL in an
arithmetic expression makes the whole expression NULL, so one unpaid order would empty a
channel's gap cell and an empty cell reads as no problem here.

In [10]:
kit.sql_table("""
    WITH paid AS (SELECT order_id, sum(amount) AS collected FROM payments GROUP BY order_id)
    SELECT o.channel, count(*) AS orders, sum(o.amount) AS booked,
           coalesce(sum(p.collected), 0) AS collected,
           sum(o.amount) - coalesce(sum(p.collected), 0) AS gap
    FROM orders o LEFT JOIN paid p ON p.order_id = o.order_id
    WHERE o.quarter = 'Q2' GROUP BY o.channel ORDER BY gap DESC
""", caption="Q2 booked against collected, by channel", conn=conn)

channel,orders,booked,collected,gap
store,159,"32,148,730","31,196,760","951,970"
web,150,"23,661,000","22,879,290","781,710"
app,153,"42,590,270","42,589,770",500


In [11]:
kit.matrix(["never paid", "refunded", "retried"],
           ["what it does to the gap", "whose problem"],
           [["widens it", "collections"], ["widens it", "operations"],
            ["narrows it, wrongly", "the platform team"]],
           title="Three things inside one number")
kit.check_summary()